# Tahap 05: Data Training (Model Utama)

Tahap ini berfokus pada pelatihan model klasifikasi menggunakan algoritma Multinomial Logistic Regression. Proses diawali dengan pemuatan matriks fitur yang telah diseleksi, pemetaan label kelas teks (Keluhan, Saran, Pujian) ke dalam format numerik diskrit, dan dilanjutkan dengan penyesuaian bobot kelas untuk menangani masalah ketidakseimbangan data (*imbalanced data*).

In [ ]:
# --- LIBRARIES ---
import os
import pickle
import pandas as pd
from scipy import sparse
from sklearn.linear_model import LogisticRegression

# --- 1. KONFIGURASI DIREKTORI & PATH ---
BASE_DIR = r'C:\Users\Asus\Desktop\SKRIPSI\1. MAIN'

print("="*60)
print("PROSES TRAINING: MULTINOMIAL LOGISTIC REGRESSION")
print("="*60)

# --- 2. PEMUATAN DATA MATRIKS DAN LABEL ---
# Memuat matriks fitur yang telah melalui tahap Feature Selection (Chi-Square)
X_train_selected = sparse.load_npz(os.path.join(BASE_DIR, 'X_train_selected_2.0.npz'))

# Memuat data tabular untuk mengambil kolom label
train_df = pd.read_csv(os.path.join(BASE_DIR, 'data_train_final_2.0.csv'))

# --- 3. PEMETAAN KELAS (CUSTOM LABEL MAPPING) ---
# Mengonversi label kategorikal menjadi nilai numerik diskrit
custom_mapping = {'keluhan': 0, 'saran': 1, 'pujian': 2}
y_train = train_df['label_pks'].map(custom_mapping)

print("Custom Mapping Berhasil Diterapkan!")
print(f"Detail Mapping: {custom_mapping}")

# Validasi kelengkapan pemetaan label
if y_train.isnull().any():
    print("\nPERINGATAN: Terdapat label yang gagal dipetakan (terdapat nilai Null)!")
    print("Daftar label unik pada dataset CSV:", train_df['label_pks'].unique())
    print("Pastikan penulisan kunci pada 'custom_mapping' sinkron dengan data CSV (case-sensitive).")
else:
    print("Validasi sukses: Seluruh data label ter-mapping dengan sempurna.")
print("-" * 60)

## 5.1 Inisialisasi dan Pelatihan Model
Konfigurasi *hyperparameter* pada *Logistic Regression* menggunakan parameter `multi_class='multinomial'` untuk klasifikasi lebih dari dua kelas, `solver='lbfgs'` sebagai algoritma optimasi, dan `class_weight='balanced'` untuk memberikan bobot penalti secara otomatis pada kelas minoritas.

In [ ]:
# --- 4. KONFIGURASI & TRAINING MODEL ---
# Inisialisasi model Logistic Regression
model_lr = LogisticRegression(
    multi_class='multinomial',  # Klasifikasi untuk > 2 kelas (Pujian, Keluhan, Saran)
    solver='lbfgs',             # Algoritma optimasi standar untuk kasus multinomial
    max_iter=1000,              # Menaikkan batas iterasi agar model konvergen
    class_weight='balanced',    # Menangani ketidakseimbangan proporsi data antar kelas
    random_state=42             # Mengunci seed acak agar hasil evaluasi selalu konsisten
)

print("Sedang melatih model Logistic Regression...")
# Proses fitting data latih (X) terhadap target (y)
model_lr.fit(X_train_selected, y_train)

print("Model berhasil dilatih secara optimal!")
print("-" * 60)

# --- 5. PENYIMPANAN MODEL & MAPPING ---
# Menyimpan objek model agar dapat digunakan kembali pada tahap evaluasi
with open(os.path.join(BASE_DIR, 'model_multinomial_lr_2.0.pkl'), 'wb') as f:
    pickle.dump(model_lr, f)

# Menyimpan kamus pemetaan untuk sinkronisasi saat inversi label
with open(os.path.join(BASE_DIR, 'custom_mapping_2.0.pkl'), 'wb') as f:
    pickle.dump(custom_mapping, f)

print("Data Model dan Mapping telah diekspor dan tersimpan di folder MAIN.")
print("="*60)